# Know Your Micro-Watershed

Read a micro-watershed’s basic details, compare its terrain and see its water connections.

Run the cells in order. Change the place, identifier or columns to explore other records. Downloads from GeoLibre use your selected tehsil; these templates start with Hilsa, Nalanda, Bihar.


## Set up Python

Run the collapsed setup cells. They import the libraries and define `read_json`, a small response reader. It reads JSON text, treats non-standard `NaN` and `Infinity` numbers as missing, and also accepts JSON returned inside a string. HTTP errors and malformed responses remain visible. Expand the cells to read the code.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geopandas", "matplotlib", "requests", "pyodide-http"])
    import pyodide_http
    pyodide_http.patch_all()

import os
import re
import ast
import json
from getpass import getpass
from inspect import isawaitable
from urllib.parse import urljoin
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, FileLink
pd.set_option("display.max_colwidth", 160)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})


In [ ]:
API_URL = 'https://geoserver.core-stack.org/api/v1/'
STAC_URL = 'https://spatio-temporal-asset-catalog.s3.ap-south-1.amazonaws.com/CorestackCatalogs_merged_collection/tehsil_wise/catalog.json'
YEARS = list(range(2017, 2025))


In [ ]:
"""Small response reader embedded in the notebooks' collapsed setup cell."""
import json


def read_json(response):
    """Read JSON text; represent non-standard NaN/Infinity values as missing."""
    response.raise_for_status()
    raw_text = response.text.lstrip("\ufeff")
    try:
        # Some API tables contain bare NaN or Infinity, which are not JSON numbers.
        data = json.loads(raw_text, parse_constant=lambda value: None)
        # Also accept a JSON document returned as a JSON-encoded string.
        if isinstance(data, str):
            data = json.loads(data.lstrip("\ufeff"), parse_constant=lambda value: None)
        return data
    except ValueError as error:
        raise ValueError(
            "The server response is not readable JSON. "
            "Inspect response.status_code and response.text[:500], then retry the request."
        ) from error


## Choose the location

These three fields contain the selected tehsil when downloaded from GeoLibre. Edit them to explore another location, then restart the kernel and run from the top.


In [ ]:
state = "Bihar"
district = "Nalanda"
tehsil = "Hilsa"


## Set your API key

The [public API guide](https://docs.core-stack.org/use-precomputed-data/public-apis/) explains registration and keys. This cell reuses `CORE_STACK_API_KEY` or asks privately and stores it in this kernel’s environment. The request header is `X-API-Key`.


In [ ]:
place = {key: re.sub(r"[\s_]+", "_", value.replace("(", "").replace(")", "")).strip("_").lower()
         for key, value in {"state": state, "district": district, "tehsil": tehsil}.items()}
api_key = os.environ.get("CORE_STACK_API_KEY", "").strip()
if not api_key:
    api_key = getpass("CoRE Stack API key: ")
    if isawaitable(api_key):
        api_key = await api_key
os.environ["CORE_STACK_API_KEY"] = str(api_key).strip()
api_headers = {"X-API-Key": os.environ["CORE_STACK_API_KEY"]}


## Read one table and choose a record

The tehsil API has no table or column filter. This cell reads it once and selects a few fields. For other tables, use `pd.DataFrame(api_data[table_name])` and select `columns` from that table; the response is already in memory. The field list shows the available names.


In [ ]:
response = requests.get(API_URL + "get_tehsil_data/", params=place, headers=api_headers, timeout=180)
api_data = read_json(response)
display(pd.DataFrame({"table": list(api_data), "rows": [len(rows) for rows in api_data.values()]}))
table_name = 'mws'
table = pd.DataFrame(api_data[table_name])

display(pd.DataFrame({"field": table.columns}))

examples = table

display(examples[['uid']].head(10))
mws_id = str(examples.iloc[0]["uid"])  # Replace with an identifier from the table.
selected = table.loc[table["uid"].astype(str) == mws_id].iloc[0]
columns = ['uid', 'area_in_ha', 'watershed_code', 'basin_code', 'sub_basin_code']
display(selected.reindex(columns).to_frame("value"))


## Discover data and descriptions in STAC

STAC lists published datasets, field descriptions, downloads and styles. Change `dataset` to another item from the collection. Asset links are used as published, wherever the files are hosted. STAC describes asset fields; API tables may use different names and units, which are shown explicitly in the examples below.


In [ ]:
collection_url = urljoin(STAC_URL, "{state}/{district}/{tehsil}/collection.json".format(**place))
response = requests.get(collection_url, timeout=90)
collection = read_json(response)
items = pd.DataFrame([{"Item": link["href"].split("/")[-1].removesuffix(".json"),
                       "URL": urljoin(collection_url, link["href"])}
                      for link in collection["links"] if link["rel"] == "item"], columns=["Item", "URL"])
# Follow a relevant item link from the collection.
dataset = "terrain_vector"
matches = items.loc[items["Item"].str.endswith("_" + dataset)]
item = None
field_notes = pd.DataFrame(columns=["name", "type", "description"])
if not matches.empty:
    item_url = matches.iloc[0]["URL"]
    response = requests.get(item_url, timeout=90)
    item = read_json(response)
    display(pd.DataFrame([item["properties"]]).reindex(columns=["title", "description", "start_datetime", "end_datetime"]).T)
    field_notes = pd.DataFrame(item["properties"].get("table:columns", []))
    display(field_notes.reindex(columns=["name", "type", "description"]).head(12))
    print("Published field count:", len(field_notes), "— use field_notes to see them all.")
    display(pd.DataFrame(item["assets"]).T.reindex(columns=["title", "type", "href"]))
else:
    print("This dataset is not listed in the tehsil's STAC collection. Available items:")
    display(items)


## Compare terrain shares

The API gives terrain shares as percentages. The table keeps those field names and adds the corresponding STAC field and published description. A doughnut is used only for complete shares totalling about 100%.


In [ ]:
terrain = pd.DataFrame(api_data["terrain"]).set_index("uid").reindex([mws_id]).iloc[0]
stac_fields = {"plain_area_percent": "plain_area", "slopy_area_percent": "slopy_area", "hill_slope_area_percent": "hill_slope", "ridge_area_percent": "ridge_area", "valley_area_percent": "valley_are"}
terrain_values = pd.to_numeric(terrain.reindex(stac_fields), errors="coerce").to_frame("value")
terrain_values["stac_field"] = pd.Series(stac_fields)
terrain_values = terrain_values.join(field_notes.set_index("name")[["description", "type"]], on="stac_field")
display(terrain_values)
shares = terrain_values["value"]
if shares.notna().all() and (shares >= 0).all() and abs(shares.sum() - 100) < 0.5:
    shares.plot.pie(figsize=(9, 5), autopct="%1.1f%%", ylabel="", wedgeprops={"width": 0.45})
else:
    shares.dropna().plot.barh(figsize=(10, 3), xlabel="MWS area (%)")
plt.title(f"Terrain · {mws_id}")
plt.tight_layout()
plt.show()


## Inspect another MWS table

Choose `table_name` and `columns`; this uses the response already loaded. Here, elevation is the example. Keep the same `mws_id` to compare records across tables.


In [ ]:
table_name = "dem"
columns = ["min_elevation_in_m", "mean_elevation_in_m", "max_elevation_in_m"]
details = pd.DataFrame(api_data[table_name])
display(pd.DataFrame({"field": details.columns}))
record = details.loc[details["uid"].astype(str) == mws_id]
display(record.reindex(columns=columns).T)


### Try another field

For drainage, use `table_name = "drainage_density"` and columns `drainage_density_weighted_in_km_per_km2`, `drainage_density_std_in_km_per_km2`, `stream_order_length_in_km`. For stream orders, use `"stream_order"` and `order_1_area_percent` through `order_11_area_percent`. `river` has `river_name`; `canal` has `canal_name` and `project_name`; `mws_intersect_swb` has `swb_uid`. The village intersection table uses `mws uid` instead of `uid`.


## See upstream and downstream connections

Read the identifiers from the connectivity table and boundaries from `get_mws_geometries`. The added `connection` column describes the map colours; source fields are retained. A missing record does not mean there are no water connections.


In [ ]:
connections = pd.DataFrame(api_data["mws_connectivity"])
match = connections.loc[connections["uid"].astype(str) == mws_id]
upstream, downstream = [], []
if not match.empty:
    connection = match.iloc[0]
    upstream = ast.literal_eval(connection["upstream_mws"]) if pd.notna(connection["upstream_mws"]) else []
    downstream = [connection["downstream_mws"]] if pd.notna(connection["downstream_mws"]) and connection["downstream_mws"] else []
    display(match[["uid", "upstream_mws", "downstream_mws"]])
else:
    print("No connectivity record was returned for this MWS.")
response = requests.get(API_URL + "get_mws_geometries/", params=place, headers=api_headers, timeout=180)
boundaries = gpd.GeoDataFrame.from_features(read_json(response)["features"], crs="EPSG:4326")
boundaries["uid"] = boundaries["uid"].astype(str)
related = boundaries.loc[boundaries["uid"].isin(upstream + downstream + [mws_id])].copy()
related["connection"] = "selected"
related.loc[related["uid"].isin(upstream), "connection"] = "upstream"
related.loc[related["uid"].isin(downstream), "connection"] = "downstream"
if not related.empty:
    related.plot(column="connection", legend=True, edgecolor="white", figsize=(6, 5))
    plt.axis("off")
    plt.show()
display(pd.DataFrame({"uid_outside_returned_boundaries": sorted(set(upstream + downstream) - set(boundaries["uid"]))}))


## Look up indicators and a report

`get_mws_kyl_indicators` reads one MWS. Change `columns` to any field in `indicators.columns`. `get_mws_report` returns a report link when available.


In [ ]:
response = requests.get(API_URL + "get_mws_kyl_indicators/", params={**place, "mws_id": mws_id}, headers=api_headers, timeout=180)
indicators = pd.json_normalize(read_json(response))
display(pd.DataFrame({"field": indicators.columns}))
columns = list(indicators.columns[:8])
display(indicators[columns].T)
response = requests.get(API_URL + "get_mws_report/", params={**place, "mws_id": mws_id}, headers=api_headers, timeout=90)
report = read_json(response) if response.ok else {"status": response.status_code, "detail": response.text[:500]}
display(pd.json_normalize(report))


## Find the MWS at a coordinate

Use a point inside the selected boundary, or edit `latitude` and `longitude` to look up another point.


In [ ]:
selected_boundary = boundaries.loc[boundaries["uid"] == mws_id]
if not selected_boundary.empty:
    point = selected_boundary.geometry.iloc[0].representative_point()
    coordinates = {"latitude": point.y, "longitude": point.x}
    response = requests.get(API_URL + "get_mwsid_by_latlon/", params=coordinates, headers=api_headers, timeout=90)
    result = read_json(response) if response.ok else {"status": response.status_code, "detail": response.text[:500]}
    display(pd.json_normalize(result))
